# Notebook 02 — Exploratory Data Analysis (EDA)

**Purpose:** Visualise the dataset to understand patterns, seasonality, and shock impacts.
All plots saved to `results/plots/`.

---

## 1. Imports & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import os

os.makedirs('../results/plots', exist_ok=True)
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'arrivals':'#2F5A82','shock':'#E74C3C','exchange':'#F39C12','trends':'#27AE60'}

df = pd.read_csv('../data/processed/tourism_dataset_final.csv', parse_dates=['date'])
print(df.shape)
df.head(3)

## 2. Helper: Shade Shock Periods on Any Axes

In [ ]:
def shade_shocks(ax, df):
    """Add red shaded bands wherever shock_flag==1."""
    in_shock = False
    start = None
    for _, row in df.iterrows():
        if row['shock_flag'] == 1 and not in_shock:
            start = row['date']
            in_shock = True
        elif row['shock_flag'] == 0 and in_shock:
            ax.axvspan(start, row['date'], alpha=0.15, color='red', label='Shock period')
            in_shock = False
    if in_shock:
        ax.axvspan(start, df['date'].max(), alpha=0.15, color='red')
    return ax

## 3. Monthly Arrivals Time Series (with Shock Shading)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df['date'], df['tourist_arrivals']/1000, color=COLORS['arrivals'], linewidth=1.8, label='Monthly arrivals')
shade_shocks(ax, df)

# Annotate major shocks
shock_labels = {
    '2019-04-01': 'Easter\nAttacks',
    '2020-03-01': 'COVID-19\nPandemic',
    '2022-04-01': 'Economic\nCrisis'
}
for date_str, label in shock_labels.items():
    d = pd.to_datetime(date_str)
    if d in df['date'].values:
        y = df.loc[df['date']==d, 'tourist_arrivals'].values[0] / 1000
        ax.annotate(label, xy=(d, y), xytext=(0, 40), textcoords='offset points',
                    ha='center', fontsize=8, color='#C0392B',
                    arrowprops=dict(arrowstyle='->', color='#C0392B', lw=1))

ax.set_xlabel('Date')
ax.set_ylabel('Arrivals (thousands)')
ax.set_title('Sri Lanka Monthly Tourist Arrivals 2015–2025', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../results/plots/01_arrivals_timeseries.png', dpi=150)
plt.show()
print('Saved: 01_arrivals_timeseries.png')

## 4. Seasonal Decomposition

In [ ]:
# Use only the stable (non-shock) period for clean decomposition
# Fill shock-period gaps with interpolation for decomposition only
arrivals_full = df.set_index('date')['tourist_arrivals'].astype(float)
arrivals_interp = arrivals_full.copy()
arrivals_interp[df.set_index('date')['shock_flag']==1] = np.nan
arrivals_interp = arrivals_interp.interpolate(method='linear')

decomp = seasonal_decompose(arrivals_interp, model='additive', period=12, extrapolate_trend='freq')

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
for ax, data, title, color in zip(axes,
    [arrivals_interp, decomp.trend, decomp.seasonal, decomp.resid],
    ['Observed', 'Trend', 'Seasonal', 'Residual'],
    [COLORS['arrivals'], '#8E44AD', COLORS['trends'], '#95A5A6']):
    ax.plot(data.index, data.values/1000, color=color, linewidth=1.5)
    ax.set_ylabel(f'{title}\n(000s)', fontsize=9)
    shade_shocks(ax, df)

axes[0].set_title('Seasonal Decomposition of Tourist Arrivals', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/plots/02_seasonal_decomposition.png', dpi=150)
plt.show()
print('Saved: 02_seasonal_decomposition.png')

## 5. Correlation Heatmap

In [ ]:
feature_cols = [
    'tourist_arrivals', 'usd_lkr_rate_monthly',
    'gtrends_flights', 'gtrends_hotels', 'gtrends_visa',
    'shock_flag', 'arrivals_lag_1', 'arrivals_lag_3', 'arrivals_lag_12'
]
corr = df[feature_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/plots/03_correlation_heatmap.png', dpi=150)
plt.show()
print('Saved: 03_correlation_heatmap.png')

## 6. Exchange Rate & Google Trends vs Arrivals

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Panel A: Exchange rate
ax1 = axes[0]
ax1b = ax1.twinx()
ax1.plot(df['date'], df['tourist_arrivals']/1000, color=COLORS['arrivals'], label='Arrivals (L)')
ax1b.plot(df['date'], df['usd_lkr_rate_monthly'], color=COLORS['exchange'], alpha=0.8, label='USD/LKR (R)')
ax1.set_ylabel('Arrivals (000s)', color=COLORS['arrivals'])
ax1b.set_ylabel('USD/LKR', color=COLORS['exchange'])
ax1.set_title('Tourist Arrivals vs USD/LKR Exchange Rate', fontweight='bold')
shade_shocks(ax1, df)

# Panel B: Google Trends — flights
axes[1].plot(df['date'], df['tourist_arrivals']/1000, color=COLORS['arrivals'], label='Arrivals')
ax2b = axes[1].twinx()
ax2b.plot(df['date'], df['gtrends_flights'], color=COLORS['trends'], alpha=0.8, label='Trends: Flights')
axes[1].set_ylabel('Arrivals (000s)', color=COLORS['arrivals'])
ax2b.set_ylabel('Search Interest', color=COLORS['trends'])
axes[1].set_title('Tourist Arrivals vs Google Trends (Flights)', fontweight='bold')
shade_shocks(axes[1], df)

# Panel C: Google Trends — hotels
axes[2].plot(df['date'], df['tourist_arrivals']/1000, color=COLORS['arrivals'], label='Arrivals')
ax3b = axes[2].twinx()
ax3b.plot(df['date'], df['gtrends_hotels'], color='#9B59B6', alpha=0.8, label='Trends: Hotels')
axes[2].set_ylabel('Arrivals (000s)', color=COLORS['arrivals'])
ax3b.set_ylabel('Search Interest', color='#9B59B6')
axes[2].set_title('Tourist Arrivals vs Google Trends (Hotels)', fontweight='bold')
shade_shocks(axes[2], df)

plt.tight_layout()
plt.savefig('../results/plots/04_exogenous_overlays.png', dpi=150)
plt.show()
print('Saved: 04_exogenous_overlays.png')

## 7. ACF & PACF Plots (for SARIMA order selection)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(arrivals_interp,  lags=36, ax=axes[0], title='ACF — Tourist Arrivals')
plot_pacf(arrivals_interp, lags=36, ax=axes[1], title='PACF — Tourist Arrivals')
plt.tight_layout()
plt.savefig('../results/plots/05_acf_pacf.png', dpi=150)
plt.show()
print()
print('Interpretation guide:')
print('  ACF: Significant spike at lag 12 => strong annual seasonality (m=12 for SARIMA)')
print('  PACF: Number of significant spikes => AR order suggestion')
print('Saved: 05_acf_pacf.png')

## 8. Boxplot: Monthly Seasonality Pattern

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

# Stable periods only for cleaner seasonality
stable = df[df['shock_flag']==0].copy()
sns.boxplot(data=stable, x='month', y='tourist_arrivals',
            palette='Blues', ax=ax, order=range(1,13))
ax.set_xticklabels(month_names)
ax.set_xlabel('Month')
ax.set_ylabel('Arrivals')
ax.set_title('Monthly Seasonality Pattern (Stable Periods Only)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/plots/06_monthly_seasonality.png', dpi=150)
plt.show()
print('Saved: 06_monthly_seasonality.png')

## 9. EDA Summary

In [ ]:
print('=== EDA COMPLETE ===')
print()
print('Key observations to carry forward:')
print('  1. Strong annual seasonality (Dec-Mar peak, May-Sep trough)')
print('  2. Three distinct shock periods visible — Easter 2019, COVID 2020-21, Crisis 2022')
print('  3. Google Trends (flights) appears to lead arrivals by ~1-2 months')
print('  4. Exchange rate shows strong correlation in crisis period')
print()
print('Plots saved to: results/plots/')
print('DONE  Proceed to 03_sarima.ipynb')